In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "/workspaces/dev/modules/python-utils",
    "/workspaces/dev/modules/ai-utils",
    "/workspaces/dev/test/performance_test/esic",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from pathlib import Path

In [ ]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.esic_v1 import search_all_ref_and_hyp
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.audio_utils import *
from sj_utils.string_utils import *
from sj_utils.collection_utils import SafetyDict
from sj_utils.evaluator import TimeChecker

In [ ]:
from util import get_token_saver_loader_transcriber, normalize_text

In [ ]:
SOURCE = "/workspaces/dev/datasets/ESIC-v1.1/v1.1/test"
STORAGE = "/workspaces/dev/storage/esic/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000
HYPERPARAMETERS_PATH = "/workspaces/dev/hyperparameters/esic/20250720_wer7o3.yml"
OVERLAP = 96000

In [ ]:
src = Path(SOURCE)
storage = Path(STORAGE)

In [ ]:
hyperparameters = SafetyDict({
    "whisper": {
        "model_options": {
            "model_size_or_path": "large-v3",
            "device": "cuda",
            "compute_type": "float16",
        },
        "transcribe_options": {
            "beam_size":5,
            "vad_filter": False,
            "temperature": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
        }
    },
    "silero_vad": {
        "model_options": {},
        "run_options": {}
    },
    "asr": {
        "max_overlap_duration": 16000,
    },
    "position_weighted_filter": {
        "boundary": 0,
    },
    "duration_filter": {
        "z_thresh": {
            "default": 2.0,
            "ko": 2.0,
            "en": 5.0,
        },
        "min_dur": {
            "default": 160,
            "ko": 160,
            "en": 160,
        }
    },
    "probability_filter":{
        "z_thresh":{
            "default": 3.0,
            "ko": 3.0,
            "en": 3.4,
        },
        "min_prob": {
            "default": 1.0,
            "ko": 0.4,
            "en": 0.15
        },
    },
    "selector":{
        "iou_threshold": {
            "default": 0.5,
            "ko": 0.4,
            "en": 0.75,
        },
        "cos_threshold":{
            "default": 0.5,
            "ko": 0.25,
            "en": 0.64
        },
        "padding": {
            "default": 3200,
            "ko": 3200,
            "en": 15800
        },
    },
    "max_overlap_duration": 96000
})

In [ ]:
transcribe_time = TimeChecker()
processed_time = TimeChecker()

In [ ]:
_transcriber = get_token_saver_loader_transcriber(src, storage, SAMPLE_RATE)
# transcriber = lambda audio: _transcriber(audio, transcribe_time, HYPERPARAMETERS_PATH, OVERLAP)
transcriber = lambda audio: _transcriber(audio, transcribe_time, hyperparameters, hyperparameters["asr"]["max_overlap_duration"])

In [ ]:
processed_time.start()
data = search_all_ref_and_hyp(src, transcriber, normalize_text, 2)
processed_time.check()

In [ ]:
concat_result = {}
for value in data.values():
    for k, v in value.items():
        if k not in concat_result:
            concat_result[k] = []
        concat_result[k].append(v)

In [ ]:
output = sclite_trn(
    concat_result["ref"],
    concat_result["hyp"],
)

In [ ]:
{
    "result":parse_sclite_summary(output),
    "processed_time": processed_time.metric(),
    "transcribe_time": transcribe_time.metric(),
}